# 00b · Setup and hardware

**The decision this notebook supports:** what can this machine actually train?

The lab was built and tested on an Apple Silicon laptop with 24 GB of unified memory, using PyTorch's MPS backend. `lab.device()` also selects CUDA or CPU automatically, but only the Apple Silicon path has been tested. Establish your limits up front rather than discovering them halfway through a training run.

Run notebook **00a** first, so the base model is on disk. Nothing here costs money.

In [ ]:
# Load this project's shared paths, configuration, scoring, and file helpers.
# sys controls Python's import search path; adding '..' assumes the notebook folder
# is the working directory. Import the project root, not src itself, so src/secrets.py
# does not replace Python's own secrets module. warnings controls diagnostic messages;
# the existing filter hides warnings, which can also hide useful compatibility clues.
# json handles JSON data; it is imported here for later cells (unused in this setup).
# CFG is a nested dictionary: CFG["model"]["name"] reads its model-name setting.
import sys, json, warnings
sys.path.insert(0, "..")          # project root, so `from src import lab` works
warnings.filterwarnings("ignore") # hide library deprecation noise
from src import lab
CFG = lab.cfg()
print("lab ready |", CFG["model"]["name"])

## What is on this machine

Two compute paths exist on Apple Silicon and this lab uses both:

- **MLX** is Apple's own array framework. It is native, it uses unified memory directly, and it is fastest for inference and quantisation here.
- **PyTorch with the MPS backend** talks to the same GPU through Metal. The Hugging Face training stack, meaning `transformers`, `peft` and `trl`, runs on it. We use this for the actual training because TRL's `DPOTrainer` lives here.

Unified memory is the thing worth understanding. On a discrete-GPU machine, model weights must be copied from system RAM into separate VRAM. Here there is one pool, so a 0.5B model in float32 occupies roughly 2 GB and everything else you have open competes for the same memory.

In [ ]:
# Check the available execution backends before choosing training settings.
# platform is Python's standard system-information library; torch is PyTorch,
# which runs tensor calculations and the Hugging Face training stack.
# is_built() means this PyTorch build includes MPS support; is_available() also
# checks whether this machine can use it now. Neither call loads model weights.
import torch, platform
print("machine        :", platform.machine())
print("python         :", platform.python_version())
print("torch          :", torch.__version__)
print("MPS available  :", torch.backends.mps.is_available())
print("MPS built      :", torch.backends.mps.is_built())

# mlx.core is Apple's tensor library, imported under the short name mx.
# We inspect its default device only; later training uses PyTorch rather than MLX.
import mlx.core as mx
print("MLX device     :", mx.default_device())

## Memory arithmetic, done by hand first

Before trusting any library, work out what should fit. For a model with `P` parameters:

- **Inference** in float32 needs about `4P` bytes for the weights, plus a KV cache that grows with sequence length.
- **Full fine-tuning** needs roughly `4P` for weights, `4P` for gradients, and `8P` for the Adam optimiser state, so about `16P` in total.
- **LoRA** freezes the base weights, so you need `4P` for the frozen model plus optimiser state for only the small adapter matrices. That is why LoRA fits where full fine-tuning does not.

Compute it for Qwen2.5-0.5B and check the claim.

In [ ]:
# Estimate memory before loading a model. Underscores make 494_000_000 readable
# without changing its numeric value. lambda defines a short function: gb(b)
# divides bytes by 1024**3, where ** means exponentiation. These are GiB although
# the existing print labels say GB. This estimate omits activations, KV cache,
# temporary buffers, and framework overhead, so it is not a peak-memory guarantee.
P = 494_000_000   # Qwen2.5-0.5B, approximately
gb = lambda b: b / 1024**3

# An f-string evaluates expressions inside braces. :5.2f prints a decimal
# with two digits after the point; :, below adds thousands separators.
print(f"inference  fp32 weights      : {gb(4*P):5.2f} GB")
print(f"full FT    (weights+grad+opt): {gb(16*P):5.2f} GB")

# LoRA: rank-r adapters on the attention and MLP projections only
# Tuple unpacking assigns four settings at once. A rank-r update uses two
# small matrices instead of updating the full projection. The formula below
# assumes every target is hidden-by-hidden; Qwen's K/V and MLP dimensions differ,
# so this is a toy estimate. PEFT's trainable-parameter count is the real check.
r, hidden, n_layers, n_targets = 16, 896, 24, 7
lora_params = n_layers * n_targets * 2 * r * hidden   # A and B matrices
print(f"LoRA trainable params        : {lora_params:,}  ({100*lora_params/P:.2f}% of base)")
print(f"LoRA   (frozen + adapter opt): {gb(4*P + 16*lora_params):5.2f} GB")

That last number is the whole reason this lab is possible on a laptop. Note it: you will be asked in an interview why LoRA rather than full fine-tuning, and *"it fits in memory"* is a better answer when you can state the ratio.

## The model, and your API keys

Notebook 00a downloaded the base model into `models/`. The cell below checks it is there, without loading the weights.

API keys are **optional**: only the Gemini and OpenAI cells in notebooks 03, 09, 10 and 12 use them. `lab.load_secret` reads a key from an environment variable, or from a git-ignored `.env` file in the project root (copy `.env.example`). **Values are never printed, never written to a notebook output, and never committed.** The cell reports only which key names are present. See `docs/SETUP.md`.

In [ ]:
# pathlib.Path gives portable file/directory operations. Inspect the downloaded checkpoint
# without loading its weights. If it is missing, run notebooks/00a_download_models.ipynb first.
from pathlib import Path
p = Path(CFG["model"]["base_path"])
assert (p / "config.json").is_file(), f"Base model not found at {p}. Run notebook 00a first."
print("model path:", p)
print("files:", sorted(x.name for x in p.iterdir())[:8])
# rglob('*') visits entries recursively; stat().st_size reads stored byte sizes.
# This is a filesystem-size estimate, not the memory needed for inference.
size_gb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
print(f"on-disk size: {size_gb:.2f} GB")

# Names only, never values. An empty list is fine: API keys are optional.
print("\nAPI keys found:", lab.list_secret_names() or "none (only the optional Gemini/OpenAI cells need them)")

## Load the model once and generate

This is the slow cell the first time, roughly 20 to 40 seconds. It proves the whole inference path works before you build anything on top of it.

In [ ]:
# Load one model for all subsequent chat calls, rather than reloading per question.
# transformers provides checkpoint-aware tokenization and causal text generation;
# torch chooses tensor precision/device; time will measure generation duration.
# lab.device() picks CUDA, then Apple MPS, then CPU (configs/config.yaml: device.torch).
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, time

MODEL_PATH = CFG["model"]["base_path"]
DEVICE = lab.device()

tok = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.float32).to(DEVICE)
# eval() switches off training-time behaviors such as dropout. It does not
# turn off gradient tracking; chat() separately uses torch.no_grad() for that.
model.eval()

# parameters() iterates over weight tensors; numel() counts their scalar values.
# The generator feeds those counts to sum without building an extra list.
n_params = sum(p.numel() for p in model.parameters())
print(f"loaded on {DEVICE} | {n_params:,} parameters | {n_params/1e6:.1f}M")

In [ ]:
# Define a reusable chat helper. Default arguments supply an 80-token output cap
# and a general system instruction unless the caller provides replacements.
# The returned tuple is (decoded answer, elapsed seconds, generated-token count).
def chat(prompt, max_new_tokens=80, system="You are a helpful assistant."):
    # These dictionaries describe the two roles. The system text sets behavior;
    # the user text supplies the question. A list preserves their conversation order.
    msgs = [{"role":"system","content":system}, {"role":"user","content":prompt}]
    # apply_chat_template is a tokenizer method from Transformers. It renders the
    # saved tokenizer's chat_template with role/control tokens. tokenize=False
    # returns a string; add_generation_prompt=True adds the assistant-start marker.
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    # Calling tok(...) converts that string into input_ids and an attention mask.
    # "pt" requests PyTorch tensors, typically shaped [batch=1, prompt_length].
    ids = tok(text, return_tensors="pt").to(DEVICE)
    t0 = time.time()
    # The with block disables gradient recording, saving inference memory.
    # **ids unpacks the dictionary into generate(input_ids=..., attention_mask=...).
    # Greedy do_sample=False chooses the highest-scoring next token each step.
    # max_new_tokens limits added tokens; an EOS token can stop generation earlier.
    # pad_token_id tells generation which ID to use if padding is required.
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    dt = time.time() - t0
    # For the default tensor return, output contains the prompt plus new tokens.
    # [0] selects the first batch row; [prompt_length:] removes the prompt.
    # Timing here is approximate: MPS work is asynchronous and no synchronization
    # was added to this original helper before recording dt.
    new = out[0][ids["input_ids"].shape[1]:]
    # decode maps IDs back to text; skip_special_tokens removes control markers.
    # strip removes outer whitespace. len(new) still counts generated special IDs.
    return tok.decode(new, skip_special_tokens=True).strip(), dt, len(new)


In [ ]:

# This call actually generates an answer. Tuple unpacking gives each returned
# value a name; n/dt is an approximate throughput measurement for this prompt.
ans, dt, n = chat("In one sentence, what is an electric scooter?")
print(f"{n} tokens in {dt:.1f}s  ({n/dt:.1f} tok/s)\n")
print(ans)

## The question this lab exists to answer

Now ask it something about Meridian Motors, the fictional company whose documents are in `data/raw/`.

The model has never seen this company. Watch what it does with the question: does it say it does not know, or does it invent something plausible? **That behaviour, not the missing fact, is the real problem**, and it is what DPO in notebook 06 will address.

In [ ]:
# Probe the base model without giving it company documents. Then read the
# reference fact from disk to compare the generated claim with the source.
# The fact lookup happens AFTER generation, so it cannot help produce the answer.
# fi is a dictionary keyed by fact ID; f-strings below display its labelled value.
ans, dt, n = chat("What is the certified range of the Meridian Arc 110 electric scooter?")
print(ans)
print("\n--- the true answer, from data/raw/facts.json ---")
fi = lab.fact_index()
f = fi["110_range_km"]
print(f"{f['label']}: {f['value']} {f['unit']}  (source: {f['source_doc']})")

## What you should be able to say now

- Why LoRA fits on this machine and full fine-tuning does not, with the ratio.
- What unified memory changes about the usual GPU memory story.
- What the stock model does when asked about something it has never seen.

**Next:** notebook 01 builds the evaluation set. Do that before any training exists. That ordering is the single most transferable idea in this lab.